# Deploy to Google Cloud Run

This notebook walks through deploying a containerized AI app to Google Cloud Run. All `gcloud` commands run in your terminal.

## 1. Prerequisites

Before deploying, ensure you have:

1. **Google Cloud account** — https://cloud.google.com/
2. **gcloud CLI installed** — https://cloud.google.com/sdk/docs/install
3. **Docker installed and running** — https://docs.docker.com/get-docker/
4. **A GCP project** with billing enabled

### Setup Commands

```bash
# 1. Authenticate with Google Cloud
gcloud auth login

# 2. Set your project
gcloud config set project YOUR_PROJECT_ID

# 3. Verify project
gcloud config get project

# 4. Enable required APIs
gcloud services enable run.googleapis.com
gcloud services enable containerregistry.googleapis.com
gcloud services enable cloudbuild.googleapis.com
```

## 2. Deployment Methods

Cloud Run offers two deployment approaches:

### Method A: Deploy from Source (Recommended)

Cloud Build builds your image and deploys in one step:

```bash
gcloud run deploy ai-backend \
  --source . \
  --platform managed \
  --region us-central1 \
  --allow-unauthenticated \
  --set-env-vars "MODEL_NAME=gemini-pro" \
  --memory 2Gi \
  --cpu 2 \
  --max-instances 10
```

**What happens**: Cloud Build reads your Dockerfile, builds the image, pushes to GCR, and deploys to Cloud Run.

### Method B: Deploy from Pre-built Image

Build locally, push to registry, then deploy:

```bash
# 1. Configure Docker for GCR
gcloud auth configure-docker

# 2. Build image locally
docker build -t gcr.io/YOUR_PROJECT_ID/ai-backend:latest .

# 3. Push to GCR
docker push gcr.io/YOUR_PROJECT_ID/ai-backend:latest

# 4. Deploy from image
gcloud run deploy ai-backend \
  --image gcr.io/YOUR_PROJECT_ID/ai-backend:latest \
  --platform managed \
  --region us-central1 \
  --allow-unauthenticated
```

## 3. Environment Configuration

### Setting Environment Variables

```bash
# Single variable
gcloud run deploy ai-backend \
  --source . \
  --set-env-vars "MODEL_NAME=gemini-pro" \
  --region us-central1

# Multiple variables
gcloud run deploy ai-backend \
  --source . \
  --set-env-vars "MODEL_NAME=gemini-pro,LOG_LEVEL=info,MAX_TOKENS=1000" \
  --region us-central1

# From a file
gcloud run deploy ai-backend \
  --source . \
  --env-vars-file .env.yaml \
  --region us-central1
```

### Using Secret Manager (Recommended for Secrets)

```bash
# 1. Create a secret in GCP Secret Manager
echo -n "your-api-key" | gcloud secrets create gemini-api-key --data-file=-

# 2. Grant Cloud Run access to the secret
gcloud secrets add-iam-policy-binding gemini-api-key \
  --member="serviceAccount:YOUR_PROJECT_NUMBER-compute@developer.gserviceaccount.com" \
  --role="roles/secretmanager.secretAccessor"

# 3. Deploy with secrets mounted as env vars
gcloud run deploy ai-backend \
  --source . \
  --set-secrets "OPENAI_API_KEY=gemini-api-key:latest" \
  --region us-central1
```

**Why Secret Manager?** Environment variables can appear in logs and process lists. Secret Manager mounts secrets as files, which is more secure.

## 4. Resource Configuration

### Configuration Flags

```bash
gcloud run deploy ai-backend \
  --source . \
  --region us-central1 \
  --memory 2Gi \
  --cpu 2 \
  --min-instances 1 \
  --max-instances 20 \
  --concurrency 80 \
  --timeout 300 \
  --set-env-vars "MODEL_NAME=gemini-pro" \
  --allow-unauthenticated
```

### When to Use What

| Scenario | CPU | Memory | Min Instances | Max Instances |
|----------|-----|--------|---------------|---------------|
| Simple API | 1 | 1Gi | 0 | 10 |
| AI inference (CPU) | 2 | 4Gi | 1 | 20 |
| AI inference (GPU) | 4 | 8Gi | 1 | 5 |
| High traffic | 2 | 2Gi | 2 | 100 |

**Note**: GPU support requires special preview access and `--gpu` flag.

## 5. Custom Domain Setup

### Map a Domain to Cloud Run

```bash
# 1. Verify domain ownership (one-time)
gcloud domains verify yourdomain.com

# 2. Create domain mapping
gcloud run domain-mappings create \
  --service ai-backend \
  --domain api.yourdomain.com \
  --region us-central1

# 3. Get the DNS records to configure
gcloud run domain-mappings describe \
  --domain api.yourdomain.com \
  --region us-central1

# 4. Add the CNAME or A record to your DNS provider
# Then wait for DNS propagation (can take up to 24 hours)
```

### Using the Auto-generated URL

Cloud Run provides a default URL:
```
https://ai-backend-XXXXXXXX-uc.a.run.app
```

This is fine for development and internal use.

## 6. Monitoring and Logs

### View Logs

```bash
# Stream logs in real-time
gcloud logging tail "resource.type=cloud_run_revision AND resource.labels.service_name=ai-backend"

# Recent logs
gcloud logging read "resource.type=cloud_run_revision AND resource.labels.service_name=ai-backend" --limit 50

# Filter by severity
gcloud logging read "resource.type=cloud_run_revision AND resource.labels.service_name=ai-backend AND severity=ERROR" --limit 20
```

### Check Service Status

```bash
# List all Cloud Run services
gcloud run services list --region us-central1

# Get service details
gcloud run services describe ai-backend --region us-central1

# Get the service URL
gcloud run services describe ai-backend --region us-central1 --format 'value(status.url)'
```

### GCP Console

For visual monitoring, visit:
- **Cloud Run**: https://console.cloud.google.com/run
- **Logs Explorer**: https://console.cloud.google.com/logs
- **Monitoring**: https://console.cloud.google.com/monitoring

## 7. Cost Management

### Pricing Breakdown

| Component | Price |
|-----------|-------|
| vCPU | $0.00002400/vCPU-second |
| Memory | $0.00000250/GiB-second |
| Requests | $0.40/million requests |
| Free tier | 240,000 vCPU-seconds, 450,000 GiB-seconds, 2M requests/month |

### Cost Optimization

```bash
# Scale to zero when idle (no traffic = no cost)
gcloud run deploy ai-backend --source . --min-instances 0 --max-instances 10

# Set a budget alert
gcloud billing budgets create \
  --billing-account=YOUR_BILLING_ACCOUNT \
  --display-name="AI Backend Budget" \
  --budget-amount=50 \
  --threshold-rule=percent=80
```

### Monitor Costs

```bash
# View current billing
gcloud billing accounts list

# Check costs in console
# https://console.cloud.google.com/billing
```

## 8. Update and Redeploy

```bash
# Update and redeploy (same command)
gcloud run deploy ai-backend \
  --source . \
  --region us-central1 \
  --set-env-vars "MODEL_NAME=gemini-pro" \
  --allow-unauthenticated

# Update specific settings without full redeploy
gcloud run services update ai-backend \
  --region us-central1 \
  --memory 4Gi

# Roll back to a previous revision
gcloud run services update-traffic ai-backend \
  --region us-central1 \
  --to-revisions=ai-backend-00001-abc=100

# Delete a service
gcloud run services delete ai-backend --region us-central1
```

## 9. Complete Deployment Checklist

Before deploying to production:

- [ ] Dockerfile builds successfully locally
- [ ] App runs in container and responds to health checks
- [ ] `.env` is in `.gitignore`
- [ ] `.env.example` documents all required variables
- [ ] Secrets are in Secret Manager (not env vars for sensitive data)
- [ ] `--max-instances` is set to prevent runaway costs
- [ ] Health check endpoint (`/health`) is implemented
- [ ] Logs are structured (JSON format for production)
- [ ] Error handling returns proper HTTP status codes
- [ ] CORS is configured if needed
- [ ] Rate limiting is considered
- [ ] Budget alerts are set up in GCP

## Summary

| Step | Command | Purpose |
|------|---------|--------|
| Auth | `gcloud auth login` | Authenticate with GCP |
| Deploy | `gcloud run deploy --source .` | Build + deploy |
| Secrets | `--set-secrets KEY=secret:version` | Mount secrets |
| Logs | `gcloud logging tail` | Monitor |
| Update | Same deploy command | Redeploy |
| Scale | `--min-instances`, `--max-instances` | Cost control |

**Next**: [Exercise 01](../exercises/exercise-01.md) — Practice deploying your own app.